# wandb-init-run — ex2: merge default + override hparams before wandb.init

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `wandb-init-run`. Running the final beacon cell reports progress against the `Logging: wandb.init run` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Logging: wandb.init run` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`wandb-init-run`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "wandb-init-run"
DD_SUBTOPIC = "Logging: wandb.init run"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `wandb.init(...)` with sweep overrides — quick refresher

Sweep agents (and Optuna / Hydra / etc.) pass per-trial hyperparameter overrides as a plain dict. The ARENA pattern is to merge them into a default args dataclass BEFORE calling `wandb.init`, so the wandb `config=` snapshot reflects the *actual* hparams used by training — not the unmodified defaults.

**Two failure modes to avoid:**

- Calling `wandb.init(config=defaults)` and THEN overriding `args.lr = sweep_lr` — the dashboard logs `defaults.lr`, not the real lr. Filtering / comparing runs becomes wrong.
- Mutating the defaults dict in place — leaks state into the next sweep trial.

**The merge recipe.** `merged = {**asdict(defaults), **overrides}`. Right-side wins. Pass `merged` (a dict) to `config=`.

### Exercise 2 — merge default + override hparams before wandb.init

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the `{**asdict(defaults), **overrides}` merge before `wandb.init(config=merged)` so the wandb config snapshot reflects the actual hparams used by the sweep trial.
> Keywords: wandb, init, sweep, config-merge, mock
> ```

**KCs targeted:** `wandb-init-config-dict`, `config-override-merge`

Implement `ex2_init_with_overrides(defaults, overrides)`. The sweep-trial wandb-init recipe:

1. `defaults` is a dataclass instance (has fields `wandb_project`, `wandb_name`, `lr`, `batch_size`, `epochs`).
2. `overrides` is a plain dict of per-trial overrides — e.g. `{'lr': 1e-4, 'batch_size': 128}`. May be empty.
3. Build the merged config: `merged = {**asdict(defaults), **overrides}` so override values WIN over defaults.
4. Call `wandb.init(project=defaults.wandb_project, name=defaults.wandb_name, config=merged)`. Note `config=` is the MERGED DICT, not the defaults object — this is the whole point.
5. Return the merged dict.

The test mocks `wandb` via `sys.modules.setdefault`, then inspects `wandb.init.call_args` to verify the merged dict landed in `config=`. Defaults must NOT be mutated (caller-side assertion).

In [ ]:
import sys
from unittest.mock import MagicMock
sys.modules.setdefault('wandb', MagicMock())
import wandb
from dataclasses import asdict

def ex2_init_with_overrides(defaults, overrides: dict) -> dict:
    merged = {**asdict(defaults), **overrides}
    wandb.init(
        project=defaults.wandb_project,
        name=defaults.wandb_name,
        config=merged,
    )
    return merged


<details><summary>Solution</summary>

```python
import sys
from unittest.mock import MagicMock
sys.modules.setdefault('wandb', MagicMock())
import wandb
from dataclasses import asdict

def ex2_init_with_overrides(defaults, overrides: dict) -> dict:
    merged = {**asdict(defaults), **overrides}
    wandb.init(
        project=defaults.wandb_project,
        name=defaults.wandb_name,
        config=merged,
    )
    return merged
```

**Right-side wins.** Python `{**a, **b}` literally builds a new dict and re-inserts `b`'s keys last — later writes overwrite earlier ones. Perfect for overrides.

**Why pass the dict, not the dataclass.** `wandb.init(config=...)` snapshots WHATEVER you give it. Pass the dataclass and wandb sees the unmodified defaults; pass the merged dict and wandb sees the true trial config. The dashboard filter `lr=1e-4` will only surface this run if config['lr'] is 1e-4.

**Never mutate defaults.** If you set `defaults.lr = overrides['lr']`, the next sweep trial inherits that mutation. Always rebuild a fresh dict per trial.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()